# Генерация кода и замер инференса на одной NVIDIA T4

Используем encoder-decoder Transformer, обученный с нуля на исходном датасете.
Генерируем ответы на все 1919 test-вопросов с параметрами beam=4,
max_new_tokens=128, batch_size=32 и length_penalty=0.6.
Вычисления выполняются с FP16 autocast и static KV-cache.

Среда запуска: Kaggle, GPU T4 x2, Internet для установки зависимостей.
Входные файлы берём из одного подключённого артефакта модели.
Отдельному процессу инференса задаём CUDA_VISIBLE_DEVICES=0, поэтому
вычисления используют одну T4.

Раздельно измеряем подготовку, полный процесс инференса и общее время ячеек.
Веса загружаются в GPU внутри измеряемого процесса.


## 1. Фиксируем параметры и запускаем общий таймер

Задаём параметры декодирования и ожидаемые SHA256 файлов модели.
VERIFY_REFERENCE_ARTIFACT включает сверку с финальным артефактом.
Общий таймер охватывает подготовку и последующие ячейки.


In [ ]:
import time
SESSION_STARTED = time.perf_counter()

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
ARTIFACT_OVERRIDE = None  # При необходимости: путь к распакованному inference-артефакту.
VERIFY_REFERENCE_ARTIFACT = True
EXPECTED_FILES = {'code_bundle.zip': 'dd8bb67240d31cac4df793bc7a4fecae4f94c0d2d0a5d45adaab5109bdba4d44', 'manifest.json': '0640ba3009dedc82142d990f1c1a5a6479067657ed1a73e485a5e42e284bb922', 'model.pt': '13e84e4951ee823bf5d03ca2f7e5eb1078dc7e0d040cdba97eb7e144b837b80c', 'test/data-00000-of-00001.arrow': '08303ff1e718a9c7b88e52d85a6cd8ddf323cf11f3c84f72669cc9af375a1c7b', 'test/dataset_info.json': 'bd594cbce97da17d645df06bd1b162947eb82e1ae0844b19bf9c6c7ff70e363f', 'test/state.json': 'cb2806ede335ce6e527f6b6d168dadf016af75a6963845bcb07896cf7ad5ebc7', 'tokenizer.json': '27c60831ad236373af954c059f9c2f6933121efecb6e882d1909f07a8f70d152'}
EXPECTED_SOURCE_MANIFEST_SHA256 = '57e1d572870a067e7c1f530fe061e94e6029a11cde20b23b2ca8aa07289b822e'
EXPECTED_PREDICTIONS_SHA256 = 'ce38be7dcadc86c455776183e061a0f1aba29dae80749c3e1527ad66dca18e8c'
CONFIG = dict(num_beams=4, max_new_tokens=128, batch_size=32, length_penalty=0.6)
print('Параметры:', CONFIG)

## 2. Находим входной артефакт

Находим model.pt, tokenizer.json, manifest.json, исходники и папку test.
При загрузке ZIP распаковываем его в /kaggle/working.
В test находятся исходные вопросы и row_id; ответы генерируем моделью.


In [ ]:
if ARTIFACT_OVERRIDE is not None:
    SOURCE_ARTIFACT = Path(ARTIFACT_OVERRIDE)
else:
    candidates = [p.parent for p in INPUT_ROOT.rglob('model.pt')
                  if (p.parent / 'tokenizer.json').is_file() and (p.parent / 'test').is_dir()]
    if not candidates:
        archives = []
        for path in INPUT_ROOT.rglob('*.zip'):
            with zipfile.ZipFile(path) as archive:
                names = set(archive.namelist())
                if {'model.pt', 'tokenizer.json', 'manifest.json'}.issubset(names):
                    archives.append(path)
        assert len(archives) == 1, f'Подключите один артефакт; подходящих ZIP: {len(archives)}'
        unpacked = WORKING_ROOT / 'uploaded_inference_artifact'
        unpacked.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                assert (unpacked / member.filename).resolve().is_relative_to(unpacked.resolve())
            archive.extractall(unpacked)
        candidates = [unpacked]
    assert len(candidates) == 1, f'Подключите один артефакт; найдено {len(candidates)}'
    SOURCE_ARTIFACT = candidates[0]
assert (SOURCE_ARTIFACT / 'test').is_dir()
print('Источник:', SOURCE_ARTIFACT)

## 3. Подготавливаем файлы

Копируем артефакт на локальный диск, проверяем SHA256 и разворачиваем
исходники из code_bundle.zip или распакованной папки code_bundle.
Сопоставляем исходные файлы с manifest.

Эти операции выполняются до замера инференса. Создание модели, загрузка
весов и перенос в GPU происходят в отдельном измеряемом процессе.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

ARTIFACT = WORKING_ROOT / 'hw2_artifact'
PROJECT = WORKING_ROOT / 'hw2_inference'
ARTIFACT.mkdir(parents=True, exist_ok=True)
PROJECT.mkdir(parents=True, exist_ok=True)
for name in ['model.pt', 'tokenizer.json', 'manifest.json']:
    shutil.copyfile(SOURCE_ARTIFACT / name, ARTIFACT / name)
shutil.copytree(SOURCE_ARTIFACT / 'test', ARTIFACT / 'test', dirs_exist_ok=True)
archive_path = SOURCE_ARTIFACT / 'code_bundle.zip'
if archive_path.is_file():
    shutil.copyfile(archive_path, ARTIFACT / 'code_bundle.zip')
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            assert (PROJECT / member.filename).resolve().is_relative_to(PROJECT.resolve())
        archive.extractall(PROJECT)
else:
    CODE_SOURCE = SOURCE_ARTIFACT
    if (SOURCE_ARTIFACT / 'code_bundle/bundle_manifest.json').is_file():
        CODE_SOURCE = SOURCE_ARTIFACT / 'code_bundle'
    assert (CODE_SOURCE / 'bundle_manifest.json').is_file(), 'Не найдена папка исходников code_bundle'
    bundle = json.loads((CODE_SOURCE / 'bundle_manifest.json').read_text())
    for relative in [*bundle['sha256'], 'bundle_manifest.json']:
        target = PROJECT / relative
        assert target.resolve().is_relative_to(PROJECT.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(CODE_SOURCE / relative, target)

if VERIFY_REFERENCE_ARTIFACT:
    for relative, digest in EXPECTED_FILES.items():
        candidate = ARTIFACT / relative
        # Kaggle иногда распаковывает вложенный code_bundle.zip автоматически.
        if relative == 'code_bundle.zip' and not candidate.is_file():
            continue
        assert sha256_file(candidate) == digest, f'Не совпал файл: {relative}'
if VERIFY_REFERENCE_ARTIFACT:
    assert sha256_file(PROJECT / 'bundle_manifest.json') == EXPECTED_SOURCE_MANIFEST_SHA256
for relative, digest in json.loads((PROJECT / 'bundle_manifest.json').read_text())['sha256'].items():
    assert sha256_file(PROJECT / relative) == digest, f'Не совпал исходник: {relative}'
manifest = json.loads((ARTIFACT / 'manifest.json').read_text())
assert manifest['splits']['test']['rows'] == 1919
assert sha256_file(ARTIFACT / 'tokenizer.json') == manifest['tokenizer_sha256']
assert set(json.loads((ARTIFACT / 'test/dataset_info.json').read_text())['features']) == {'question', 'row_id'}
print('Проверено: модель, исходники, токенизатор и 1919 test-вопросов.')

## 4. Устанавливаем зависимости

Устанавливаем пакеты из requirements-kaggle.txt, сохраняя CUDA-сборку
PyTorch из окружения Kaggle. Записываем время всей подготовки:
распаковки, копирования, проверки файлов и установки зависимостей.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-input',
                '--disable-pip-version-check', '-r', str(PROJECT / 'requirements-kaggle.txt')], check=True)
PREPARATION_SECONDS = time.perf_counter() - SESSION_STARTED
print(f'Подготовка: {PREPARATION_SECONDS:.3f} с')

## 5. Запускаем инференс и замеряем время

Запускаем новый Python-процесс и измеряем время до его завершения.
В интервал входят импорты, инициализация CUDA, загрузка весов,
токенизация исходных test-вопросов, encoder/decoder, декодирование
и запись JSONL.

Каждый запуск ячейки создаёт новый процесс и заново генерирует ответы.
Предыдущие файлы ответов и отчёта удаляются перед замером.


In [ ]:
OUTPUT = WORKING_ROOT / 'predictions.jsonl'
REPORT = WORKING_ROOT / 'inference_report.json'
RUN_LOG = WORKING_ROOT / 'inference_run.log'
OUTPUT.unlink(missing_ok=True)
REPORT.unlink(missing_ok=True)
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1')
command = [
    sys.executable, str(PROJECT / 'inference.py'),
    f'checkpoint={ARTIFACT / "model.pt"}', f'prepared_dir={ARTIFACT}',
    f'output_path={OUTPUT}', f'report_path={REPORT}',
    'device=cuda', 'require_t4=true', 'split=test', 'limit=null',
    'amp=true', 'cache_type=static', 'sort_by_length=true', 'compile_step=false',
    *[f'{key}={value}' for key, value in CONFIG.items()],
]
with RUN_LOG.open('w') as stream:
    started = time.perf_counter()
    try:
        subprocess.run(command, cwd=PROJECT, env=env, stdout=stream,
                       stderr=subprocess.STDOUT, check=True)
    except subprocess.CalledProcessError:
        print(RUN_LOG.read_text()[-6000:])
        raise
    process_seconds = time.perf_counter() - started
report = json.loads(REPORT.read_text())
report['external_process_seconds'] = process_seconds
report['preparation_seconds'] = PREPARATION_SECONDS
report['passes_required_benchmark'] = bool(report['passes_required_benchmark'] and process_seconds <= 180)
report['notebook_elapsed_seconds'] = time.perf_counter() - SESSION_STARTED
report['notebook_total_within_180'] = report['notebook_elapsed_seconds'] <= 180
REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(f'ВЕСЬ ПРОЦЕСС ИНФЕРЕНСА: {process_seconds:.3f} с; предел 180 с')
print(json.dumps({key: report[key] for key in [
    'gpu', 'visible_cuda_devices', 'examples', 'num_beams', 'max_new_tokens',
    'batch_size', 'length_penalty', 'dtype', 'full_seconds', 'generation_seconds',
    'external_process_seconds', 'passes_required_benchmark',
]}, indent=2, ensure_ascii=False))

## 6. Проверяем результат

Проверяем 1919 ответов, формат JSONL, параметры декодирования,
одну видимую T4 и время полного процесса не больше 180 секунд.
Записываем SHA256 ответов и сравниваем его с эталонным файлом.
Изменения библиотек могут влиять на отдельные решения, поэтому совпадение
хеша выводим отдельно от проверки времени.

PASS обозначает выполнение технических условий запуска.
Pass@1 оценивается отдельно по правильности решений.


In [ ]:
rows = [json.loads(line) for line in OUTPUT.read_text().splitlines()]
assert len(rows) == 1919 and report['expected_split_examples'] == 1919
assert all(set(row) == {'code'} and isinstance(row['code'], str) for row in rows)
assert report['full_test'] and report['tokenized_raw_questions_inside_timer']
assert report['visible_cuda_devices'] == 1 and 'T4' in report['gpu']
for key, value in CONFIG.items():
    assert report[key] == value, f'Другой параметр: {key}'
assert report['dtype'] == 'fp16_autocast' and report['cache_type'] == 'static'
assert report['sort_by_length'] and not report['compile_step']
assert report['passes_required_benchmark'] and process_seconds <= 180, 'Лимит инференса не выполнен'
report['predictions_sha256'] = sha256_file(OUTPUT)
report['reference_predictions_match'] = report['predictions_sha256'] == EXPECTED_PREDICTIONS_SHA256
report['notebook_elapsed_seconds'] = time.perf_counter() - SESSION_STARTED
report['notebook_total_within_180'] = report['notebook_elapsed_seconds'] <= 180
REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(f'PASS: 1919 задач, одна T4, полный процесс {process_seconds:.3f} с <= 180 с')
print(f'Подготовка отдельно: {PREPARATION_SECONDS:.3f} с')
print(f'Все ячейки с подготовкой: {report["notebook_elapsed_seconds"]:.3f} с')
print('Хеш совпал с эталонным файлом:', report['reference_predictions_match'])
print('Ответы:', OUTPUT)
print('Отчёт:', REPORT)
from IPython.display import FileLink, display
display(FileLink(str(OUTPUT)))
display(FileLink(str(REPORT)))

## Отчёт о запуске

Получаем predictions.jsonl с ответами и inference_report.json с параметрами
и измерениями. В отчёте сохраняются три интервала:

| Поле | Что измеряет |
| --- | --- |
| external_process_seconds | Полный отдельный процесс инференса |
| preparation_seconds | Подготовку файлов и зависимостей |
| notebook_elapsed_seconds | Время от первой ячейки, включая подготовку и паузы между ячейками |

Время инференса, приведённое в README, соответствует external_process_seconds.
Общее время ячеек показано отдельно.
